# Code to reproduce the results in the technical appendix

In [1]:
import pandas as pd
import ast 
import altair as alt

from discovery_child_development import PROJECT_DIR, S3_BUCKET, logging
from discovery_child_development.analysis.initial_results import utils
from nesta_ds_utils.loading_saving import S3

from discovery_child_development.utils import analysis_utils as au
from discovery_child_development.utils import plotting_utils as pu
from discovery_child_development.utils import chart_trends

# Remove altair warning
import altair as alt
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

S3_OUTPUTS_DIR = '2024-07-iss-child-development/outputs/'

2024-08-06 11:25:37,703 - botocore.credentials - INFO - Found credentials in environment variables.


2024-08-06 11:25:38,850 - datasets - INFO - PyTorch version 2.4.0 available.


/opt/homebrew/Caskroom/miniconda/base/envs/discovery_child_development/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TECH = 'Technology'

# Taxonomy dataframe
topics_df = utils.load_topic_data()

# List all tech major categories
tech_subtypes = set(topics_df.query("type == @TECH").subtype.unique())
print(tech_subtypes)

{'Internet', 'Immersive tech', 'AI', 'Mobile'}


## Helper functions

In [3]:
# Relevant document ids for time series
def get_tech_ids(data_exploded_df):
    """Get relevant document ids for time series/growth estimations"""
    return (
        data_exploded_df
        .query("type == @TECH")
        .query("year >= 2013")
        .drop_duplicates('id')
        .id.to_list()
)

def get_tech_ids_5y(data_exploded_df: pd.DataFrame) -> pd.DataFrame:
    """Get relevant document ids for 2019-2023 stats"""
    return (
        data_exploded_df
        .query("type == @TECH")
        .query("year >= 2019")
        .drop_duplicates('id')
        .id.to_list()
    )

def report_magnitude_growth(magnitude_growth_df: pd.DataFrame) -> None:
    """Report magnitude and growth"""
    # Smoothed growth in 2019-2023
    growth = magnitude_growth_df.growth.iloc[0]
    # Total funding in 2019-2023 (in millions)
    magnitude = magnitude_growth_df.magnitude.iloc[0] * 5 

    logging.info(f"Growth in 2019-2023: {growth:.2f}%")
    logging.info(f"Total in 2019-2023: {magnitude:.2f}")

    return growth, magnitude


def get_tech_distribution(data_exploded_df: pd.DataFrame, tech_ids_5y, values: list, column: str='subtype') -> pd.DataFrame:
    """Get distribution of technology projects"""
    return utils.get_data_distribution(
        (
            data_exploded_df
            .query('id in @tech_ids_5y')
            .query("type == @TECH")
            .drop_duplicates(['id', column])
        ),
        column=column, 
        values=values,
    ) 


def get_tech_ts(data_exploded_df: pd.DataFrame, tech_ids, values: list, column: str='subtype') -> pd.DataFrame:
    """Get distribution of technology projects"""
    return utils.get_data_distribution(
        (
            data_exploded_df
            .query('id in @tech_ids')
            .query("type == @TECH")
            .drop_duplicates(['id', column])
        ),
        column=column, 
        values=values,
        ts = True,
    )  

def get_tech_trends(data_exploded_df: pd.DataFrame, tech_ids, tech_ids_5y, values: list, column='subtype') -> pd.DataFrame:
    tech_dist = get_tech_distribution(data_exploded_df, tech_ids_5y, values, column=column)
    tech_ts = get_tech_ts(data_exploded_df, tech_ids, values, column=column)
    _value = 'counts' if values[0] == 'id' else values[0]
    tech_magnitude_growth = utils.magnitude_and_growth(
        tech_ts, 
        column = column, 
        value=_value)
    tech_trends = (
        tech_dist.merge(
            tech_magnitude_growth, 
            on=column, 
            how='left')
    )
    return tech_trends

def get_application_trends(data_exploded_df: pd.DataFrame, tech_ids, tech_ids_5y, values: list, column:str = 'type', hide_categories: list = ['Technology', 'General']) -> pd.DataFrame:
    """Get application trends"""
    tech_applications_df = utils.get_data_distribution(
        data_exploded_df.query('id in @tech_ids_5y'), 
        column=column, 
        values=values,
    ) 
    trends_df = utils.get_data_magnitude_growth(
        data_exploded_df, ids=tech_ids, 
        column=column, 
        value=values[0])  
    return (
        tech_applications_df
        .merge(trends_df.drop('counts', axis=1), on=column, suffixes=('', '_'))
        .query(f"{column} not in @hide_categories")
        ) 


def trends_chart(application_trends_df: pd.DataFrame) -> pd.DataFrame:
    return (
        alt.Chart(application_trends_df)
        .mark_point()
        .encode(
            x='magnitude:Q',
            y='growth:Q',
            color='type:N',
            tooltip=['type', 'magnitude', 'growth'],
        )
    )

## Research funding
- Fig 2: Growth and total early-years digital tech research funding in 2019-2023
- Breakdowns by funder type (top funder; proportion from Innovate UK)
- Fig 4: Proportion of early-years funding associated with digital technologies
- Fig 5: Proportion and growth of major digital tech categories in 2019-2023
- Fig 6: Proportion and growth of major application areas in 2019-2023
- Baseline funding growth across all sectors in 2019-2023


In [4]:
# Load the data
ukri_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/data_ukri_2024.csv',
        download_as='dataframe'
    )
    # Remove the instances tagged with expressive arts due to too much noise
    .query("topics != 'arts'")
)

# Explode by topics
ukri_exploded_df = utils.explode_data(ukri_df).query("topics != 'arts'")

# Get relevant document ids
ukri_tech_ids = get_tech_ids(ukri_exploded_df)
ukri_tech_ids_5y = get_tech_ids_5y(ukri_exploded_df)


In [5]:
logging.info(f"Total number of UK research projects {len(ukri_df)}")

2024-08-06 11:25:39,282 - root - INFO - Total number of UK research projects 1091


### Growth and total early-years digital tech research funding

Figure 2

In [6]:
# Filter only technology projects
ukri_tech_type_df = (
    ukri_exploded_df
    .query("id in @ukri_tech_ids")
    .drop_duplicates(['id'])
)

ukri_ts_amounts_tech = utils.get_timeseries(ukri_tech_type_df, column='amount')
ukri_ts_counts_tech = utils.get_timeseries(ukri_tech_type_df, column='id')
utils.plot_quick_ts(ukri_ts_amounts_tech, 'amount')

alt.Chart(...)

In [7]:
# Get magnitude and growth
magnitude_growth_ukri = au.ts_magnitude_growth_(
    ukri_ts_amounts_tech,
    year_start = 2019,
    year_end = 2023
)

growth, magnitude = report_magnitude_growth(magnitude_growth_ukri)

2024-08-06 11:25:39,313 - root - INFO - Growth in 2019-2023: 165.74%
2024-08-06 11:25:39,314 - root - INFO - Total in 2019-2023: 58838.91


### Breakdowns by funder type
- Top funders
- Proportion from Innovate UK

In [8]:
# Check who are the top research funders for early-years digital tech
top_funders = (
    ukri_exploded_df
    .drop_duplicates(['id', 'lead_funder'])
    .query("id in @ukri_tech_ids_5y")
    .query("year <= 2023")    
    .groupby('lead_funder')
    .agg(amount=('amount', 'sum'))
    .sort_values('amount', ascending=False)
    .assign(proportion = lambda df: df.amount / df.amount.sum())
)
top_funders.head(5)

,amount,proportion
lead_funder,,
MRC,21617.556,0.367402
ESRC,5727.531,0.097343
Innovate UK,5584.073,0.094904
BBSRC,5425.603,0.092211
FLF,4700.880,0.079894


In [9]:
# Projects funded in the past five years
_funding_df = (
    ukri_exploded_df
    .query("id in @ukri_tech_ids_5y")
    .query("year <= 2023") 
    .drop_duplicates('id')
)

# Get the total funding
funding_total = _funding_df.amount.sum()

# Get Innovate UK funding specifically
funding_innovate_uk = (
    _funding_df
    .query("lead_funder == 'Innovate UK'")
    .amount.sum())

# Calculate proportion of funding by Innovate UK
proportion_innovate_uk = funding_innovate_uk / funding_total

logging.info(f"Propotion of funding by Innovate UK: {proportion_innovate_uk:.2f}")

2024-08-06 11:25:39,335 - root - INFO - Propotion of funding by Innovate UK: 0.09


In [10]:
# Get funding excluding health-related projects (for reference)
health_ids = ukri_exploded_df.query("type == 'Health'").id.to_list()
funding_wout_health = _funding_df.query("id not in @health_ids").amount.sum()

# Calculate proportion of funding by Innovate UK excluding health
funding_innovate_uk_wout_health = (
    _funding_df
    .query("lead_funder == 'Innovate UK'")
    .query("id not in @health_ids")
    .amount.sum())

proportion_innovate_uk_wout_health = funding_innovate_uk_wout_health / funding_wout_health

logging.info(f"Propotion of funding by Innovate UK excluding health: {proportion_innovate_uk_wout_health:.2f}")

2024-08-06 11:25:39,346 - root - INFO - Propotion of funding by Innovate UK excluding health: 0.17


### Proportion of early-years funding associated with digital technologies

Figure 4

In [11]:
# Calculate the total early-years project funding
total_early_years_funding = (
    ukri_exploded_df
    .query("year >= 2019 and year <= 2023")
    .drop_duplicates('id')
    .amount.sum())

# Get the proportion of funding for early-years digital tech
proportion_early_years_tech_ukri = funding_total / total_early_years_funding
logging.info(f"Proportion of funding for early-years digital tech: {proportion_early_years_tech_ukri:.2f}")

2024-08-06 11:25:39,354 - root - INFO - Proportion of funding for early-years digital tech: 0.20


In [12]:
# Check Technology proportion using another approach
utils.get_data_distribution(
    ukri_exploded_df.query("year >= 2019 and year <= 2023"),
    column='type', 
    values=['id', 'amount']
)

,type,counts,counts_prop,amount,amount_prop
0,Biosciences,125,0.222,84393.908,0.288
1,Child care & preschool,23,0.041,16093.698,0.055
2,Development & learning,123,0.218,56456.251,0.193
3,General,348,0.618,195291.562,0.667
4,Health,362,0.643,209860.724,0.717
5,Parenting,20,0.036,8121.362,0.028
6,Society,133,0.236,68456.721,0.234
7,Technology,105,0.187,58838.909,0.201


### Digital tech trends in 2019-2023 (funding)

Figure 5

In [13]:
ukri_tech_trends = get_tech_trends(
    ukri_exploded_df.query("year <= 2023") , 
    ukri_tech_ids, 
    ukri_tech_ids_5y, 
    ['amount', 'id']
)

ukri_tech_trends

,subtype,amount,amount_prop,counts,counts_prop,type,magnitude,growth
0,AI,40910.811,0.695,62,0.59,Technology,8182.1622,147.968645
1,Immersive tech,7718.759,0.131,24,0.229,Technology,1543.7518,23.475780
2,Internet,5814.574,0.099,18,0.171,Technology,1162.9148,378.897783
3,Mobile,17869.267,0.304,23,0.219,Technology,3573.8534,87.113273


### Application area trends in 2019-2023 (funding)

Figure 6

In [14]:
ukri_application_trends = get_application_trends(
    ukri_exploded_df.query("year <= 2023") , 
    ukri_tech_ids, 
    ukri_tech_ids_5y, 
    ['amount', 'id']
)

ukri_application_trends


,type,amount,amount_prop,counts,counts_prop,magnitude,growth
0,Biosciences,13298.835,0.226,23,0.219,2659.767,65.806908
1,Child care & preschool,2590.215,0.044,8,0.076,518.043,299.955036
2,Development & learning,12640.91,0.215,31,0.295,2528.182,-51.255006
4,Health,40250.779,0.684,64,0.61,8050.1558,210.796765
5,Parenting,3059.329,0.052,9,0.086,611.8658,2541.467658
6,Society,16255.875,0.276,21,0.2,3251.175,148.675879


Trends information for the heat map.

Note that the trend typology is data-informed - we're guided by the results below - but we might also slighty adjust the final trends category in a few select cases.

In [15]:
chart_trends.estimate_trend_type(
    ukri_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)[['type', 'magnitude', 'growth', 'trend_type_suggestion']]

,type,magnitude,growth,trend_type_suggestion
0,Biosciences,2659.767,65.806908,hot*
1,Child care & preschool,518.043,299.955036,emerging
2,Development & learning,2528.182,-51.255006,dormant*
4,Health,8050.1558,210.796765,hot
5,Parenting,611.8658,2541.467658,emerging
6,Society,3251.175,148.675879,hot


Designating Dev and learning as stabilising as the magnitue is in fact quite large relatively speaking

In [16]:
fig = trends_chart(
    ukri_application_trends
    .query("type != 'Biosciences'")
)
fig

alt.Chart(...)

### Baseline funding growth across all sectors

In [17]:
gtr_df = S3.download_obj(
    bucket = S3_BUCKET,
    path_from = S3_OUTPUTS_DIR + 'gtr_texts.csv',
    download_as='dataframe'
)

In [18]:
ukri_baseline_df = utils.get_baseline_ukri(gtr_df)

trends_baseline = au.ts_magnitude_growth_(
    ts_df = ukri_baseline_df,
    year_start = 2019,
    year_end = 2023  
)
ukri_baseline_magnitude = trends_baseline.loc['amount'].magnitude
ukri_baseline_growth = trends_baseline.loc['amount'].growth
logging.info(f"UKRI baseline growth: {ukri_baseline_growth:.2f}%")

2024-08-06 11:25:44,876 - root - INFO - UKRI baseline growth: -5.05%


In [19]:
_gtr_all_projects_df = (
    gtr_df
    .assign(year = lambda df: df.start.apply(lambda x: int(x[0:4])))
    .query("year >= 2019 and year <= 2023")
)

In [20]:
ukri_all_projects_funding = _gtr_all_projects_df.dropna(subset=["leadFunder"]).amount.sum()
ukri_innovate_uk_funding = _gtr_all_projects_df.query("leadFunder == 'Innovate UK'").amount.sum()
prop_innovate_uk = (ukri_innovate_uk_funding / ukri_all_projects_funding)

logging.info(f"Proportion of Innovate UK funding: {prop_innovate_uk:.2f}")

2024-08-06 11:25:44,935 - root - INFO - Proportion of Innovate UK funding: 0.20


## Research publications

- Fig 2: Growth and total early-years digital tech publications in 2019-2023
- Fig 4: Proportion of early-years publications associated with digital technologies
- Fig 5: Proportion and growth of major digital tech categories in 2019-2023
- Fig 6: Proportion and growth of major application areas in 2019-2023
- Baseline publication growth across all sectors in 2019-2023
- Geographical distribution


In [21]:
# Load the data
openalex_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/data_openalex_2024.csv',
        download_as='dataframe'
    )
    # Remove the instances tagged with expressive arts due to too much noise
    .query("topics != 'arts'")
)

# Explode by topics
openalex_exploded_df = utils.explode_data(openalex_df).query("topics != 'arts'")

# Get relevant document ids
openalex_tech_ids = get_tech_ids(openalex_exploded_df)
openalex_tech_ids_5y = get_tech_ids_5y(openalex_exploded_df)


In [22]:
logging.info(f"Total number of publications {len(openalex_df)}")

2024-08-06 11:25:48,061 - root - INFO - Total number of publications 70096


### Growth and total early-years digital tech publications

Figure 2

In [23]:
# Filter only technology projects
openalex_tech_type_df = (
    openalex_exploded_df
    .query("id in @openalex_tech_ids")
    .drop_duplicates(['id'])
)

openalex_ts_counts_tech = utils.get_timeseries(openalex_tech_type_df, column='id')
utils.plot_quick_ts(openalex_ts_counts_tech.query("year >= 2017"), 'counts')

alt.Chart(...)

In [24]:
# Get magnitude and growth
magnitude_growth_openalex = au.ts_magnitude_growth_(
    openalex_ts_counts_tech,
    year_start = 2019,
    year_end = 2023
)

growth, magnitude = report_magnitude_growth(magnitude_growth_openalex)

2024-08-06 11:25:48,092 - root - INFO - Growth in 2019-2023: 77.95%
2024-08-06 11:25:48,093 - root - INFO - Total in 2019-2023: 2493.00


### Proportion of early-years publications associated with digital technologies

Figure 4

In [25]:
# Check Technology proportion
openalex_category_distribution_df = utils.get_data_distribution(
    openalex_exploded_df.query("year >= 2019 and year <= 2023"),
    column='type', 
    values=['id']
)

# Get the proportion of technology projects
openalex_prop = openalex_category_distribution_df.query("type == @TECH").counts_prop.iloc[0]
logging.info(f"Proportion of publications: {openalex_prop:.2f}")

# See all major categories
openalex_category_distribution_df

2024-08-06 11:25:48,149 - root - INFO - Proportion of publications: 0.06


,type,counts,counts_prop
0,Biosciences,4330,0.104
1,Child care & preschool,6065,0.146
2,Development & learning,11692,0.281
3,General,23618,0.567
4,Health,20278,0.487
5,Parenting,1353,0.032
6,Society,9569,0.23
7,Technology,2493,0.06


### Digital tech trends in 2019-2023 (publications)

Figure 5

In [26]:
openalex_tech_trends = get_tech_trends(
    openalex_exploded_df.query("year <= 2023") , 
    openalex_tech_ids, 
    openalex_tech_ids_5y, 
    ['id']
)

openalex_tech_trends

,subtype,counts,counts_prop,type,magnitude,growth
0,AI,626,0.251,Technology,125.2,91.266376
1,Immersive tech,426,0.171,Technology,85.2,77.456647
2,Internet,1043,0.418,Technology,208.6,123.214286
3,Mobile,732,0.294,Technology,146.4,35.362319


### Application area trends in 2019-2023 (publications)

Figure 6

In [27]:
openalex_application_trends = get_application_trends(
    openalex_exploded_df.query("year <= 2023"), 
    openalex_tech_ids, 
    openalex_tech_ids_5y, 
    ['id']
)

openalex_application_trends


,type,counts,counts_prop,magnitude,growth
0,Biosciences,185,0.074,37.0,65.384615
1,Child care & preschool,398,0.16,79.6,96.078431
2,Development & learning,723,0.29,144.6,90.405904
4,Health,906,0.363,181.2,59.349593
5,Parenting,102,0.041,20.4,97.297297
6,Society,429,0.172,85.8,70.930233


Trends information for the heat map

In [28]:
chart_trends.estimate_trend_type(
    openalex_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)

,type,counts,counts_prop,magnitude,growth,trend_type_suggestion
0,Biosciences,185,0.074,37.0,65.384615,emerging
1,Child care & preschool,398,0.16,79.6,96.078431,emerging*
2,Development & learning,723,0.29,144.6,90.405904,hot
4,Health,906,0.363,181.2,59.349593,hot
5,Parenting,102,0.041,20.4,97.297297,emerging
6,Society,429,0.172,85.8,70.930233,hot*


In [29]:
fig = trends_chart(openalex_application_trends)
fig

alt.Chart(...)

### Geographical distribution

In [30]:
_openalex_countries_df = (
    openalex_exploded_df
    .query('subtype in @tech_subtypes')
    .query("year <= 2023") 
    .assign(country_code = lambda df: df.country_code.apply(ast.literal_eval))
    .explode('country_code')
    .drop_duplicates(['id', 'country_code'])
)

n_total_with_codes = len(
    _openalex_countries_df
    .drop_duplicates('id')
    .query("year >= 2019")
    .query("year <= 2023") 
    .dropna(subset=['country_code'])
)

openalex_countries_df, _ = utils.get_geographical_distribution(_openalex_countries_df)
openalex_countries_df = (
    openalex_countries_df
    .assign(total = lambda df: df.magnitude*5)
    .assign(proportion = lambda df: df.total / n_total_with_codes)
    .sort_values('proportion', ascending=False)
)
openalex_countries_df.head(10)

,magnitude,growth,country_code,total,proportion
0,113.8,28.102190,US,569.0,0.276616
12,64.0,431.914894,ID,320.0,0.155566
7,38.8,38.202247,GB,194.0,0.094312
5,32.0,15.116279,AU,160.0,0.077783
14,26.0,85.714286,CN,130.0,0.063199
9,19.0,34.000000,CA,95.0,0.046184
4,12.6,182.352941,IN,63.0,0.030627
34,12.0,40.000000,ES,60.0,0.029169
3,11.6,233.333333,DE,58.0,0.028196
8,11.2,90.909091,NL,56.0,0.027224


In [31]:
openalex_US = openalex_countries_df.query("country_code=='US'").proportion.iloc[0]
logging.info(f'US proportion: {openalex_US:.2f}')

openalex_GB = openalex_countries_df.query("country_code=='GB'").proportion.iloc[0]
logging.info(f'GB proportion: {openalex_GB:.2f}')

2024-08-06 11:25:49,081 - root - INFO - US proportion: 0.28
2024-08-06 11:25:49,086 - root - INFO - GB proportion: 0.09


### Baseline growth across all sectors (publications)

In [32]:
# Load the data
openalex_baseline_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/baseline_data_openalex.csv',
        download_as='dataframe'
    )
)

In [33]:
trends_baseline = au.ts_magnitude_growth_(
    ts_df = openalex_baseline_df,
    year_start = 2019,
    year_end = 2023  
)
openalex_baseline_magnitude = trends_baseline.loc['counts'].magnitude
openalex_baseline_growth = trends_baseline.loc['counts'].growth
logging.info(f"Publication baseline growth: {openalex_baseline_growth:.2f}%")

2024-08-06 11:25:49,241 - root - INFO - Publication baseline growth: -3.55%


### Baseline growth for edtech (publications)

In [34]:
# Load the data
openalex_baseline_df_edtech = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/baseline_data_openalex_edtech.csv',
        download_as='dataframe'
    )
)

trends_baseline = au.ts_magnitude_growth_(
    ts_df = openalex_baseline_df_edtech,
    year_start = 2019,
    year_end = 2023  
)
openalex_baseline_magnitude_edtech = trends_baseline.loc['counts'].magnitude*5
openalex_baseline_growth_edtech = trends_baseline.loc['counts'].growth
logging.info(f"Edtech publications in total: {openalex_baseline_magnitude_edtech:.2f}")
logging.info(f"Edtech publication baseline growth: {openalex_baseline_growth_edtech:.2f}%")

2024-08-06 11:25:49,496 - root - INFO - Edtech publications in total: 29378.00
2024-08-06 11:25:49,497 - root - INFO - Edtech publication baseline growth: 49.10%


## Patents

- Fig 2: Growth and total early-years digital tech patents in 2019-2023
- Fig 4: Proportion of early-years patents associated with digital technologies
- Fig 5: Proportion and growth of major digital tech categories in 2019-2023
- Fig 6: Proportion and growth of major application areas in 2019-2023
- Baseline patent growth across all sectors in 2019-2023
- Geographical distribution


In [35]:
# Load the data
patents_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/data_patents_2024.csv',
        download_as='dataframe'
    )
    # Remove the instances tagged with expressive arts due to too much noise
    .query("topics != 'arts'")
)

# Explode by topics
patents_exploded_df = utils.explode_data(patents_df).query("topics != 'arts'")

# Get relevant document ids
patents_tech_ids = get_tech_ids(patents_exploded_df)
patents_tech_ids_5y = get_tech_ids_5y(patents_exploded_df)

### Growth and total early-years digital tech patents

Figure 2

In [36]:
# Filter only technology projects
patents_tech_type_df = (
    patents_exploded_df
    .query("id in @patents_tech_ids")
    .drop_duplicates(['id'])
)

patents_ts_counts_tech = utils.get_timeseries(patents_tech_type_df, column='id')
utils.plot_quick_ts(patents_ts_counts_tech, 'counts')

alt.Chart(...)

In [37]:
# Get magnitude and growth
magnitude_growth_patents= au.ts_magnitude_growth_(
    patents_ts_counts_tech,
    year_start = 2019,
    year_end = 2023
)

growth, magnitude = report_magnitude_growth(magnitude_growth_patents)

2024-08-06 11:25:50,596 - root - INFO - Growth in 2019-2023: -5.22%
2024-08-06 11:25:50,597 - root - INFO - Total in 2019-2023: 2618.00


Check stats without including China

In [38]:
# Filter only technology projects
patents_tech_type_df_wout_CN = (
    patents_exploded_df
    .query('country_code != "CN"')
    .query("id in @patents_tech_ids")
    .drop_duplicates(['id'])
)

ts_counts_tech_wout_CN = utils.get_timeseries(patents_tech_type_df_wout_CN, column='id')
utils.plot_quick_ts(ts_counts_tech_wout_CN, 'counts')

alt.Chart(...)

In [39]:
# Get magnitude and growth
magnitude_growth_patents_wout_CN = au.ts_magnitude_growth_(
    ts_counts_tech_wout_CN,
    year_start = 2019,
    year_end = 2023
)

growth, magnitude = report_magnitude_growth(magnitude_growth_patents_wout_CN)

2024-08-06 11:25:50,621 - root - INFO - Growth in 2019-2023: 20.00%
2024-08-06 11:25:50,622 - root - INFO - Total in 2019-2023: 945.00


### Proportion of early-years patents associated with digital technologies

Figure 4

In [40]:
# Check Technology proportion
patents_category_distribution_df = utils.get_data_distribution(
    patents_exploded_df.query("year >= 2019 and year <= 2023"),
    column='type', 
    values=['id']
)

# Get the proportion of technology projects
patents_prop = patents_category_distribution_df.query("type == @TECH").counts_prop.iloc[0]
logging.info(f"Proportion of technology patents: {patents_prop:.2f}")

# See all major categories
patents_category_distribution_df

2024-08-06 11:25:50,644 - root - INFO - Proportion of technology patents: 0.18


,type,counts,counts_prop
0,Biosciences,275,0.019
1,Child care & preschool,619,0.043
2,Development & learning,767,0.053
3,General,10587,0.737
4,Health,3490,0.243
5,Parenting,277,0.019
6,Society,31,0.002
7,Technology,2618,0.182


### Digital tech trends in 2019-2023 (patents)

Figure 5

In [41]:
patents_tech_trends = get_tech_trends(
    patents_exploded_df.query("year <= 2023"), 
    patents_tech_ids, 
    patents_tech_ids_5y, 
    ['id']
)

patents_tech_trends

,subtype,counts,counts_prop,type,magnitude,growth
0,AI,1637,0.625,Technology,327.4,9.630459
1,Immersive tech,874,0.334,Technology,174.8,-14.874552
2,Internet,10,0.004,Technology,2.0,-33.333333
3,Mobile,677,0.259,Technology,135.4,-42.229730


### Application area trends in 2019-2023 (patents)

Figure 6

In [42]:
patent_application_trends = get_application_trends(
    patents_exploded_df.query("year <= 2023"), 
    patents_tech_ids, 
    patents_tech_ids_5y, 
    ['id']
)

patent_application_trends

,type,counts,counts_prop,magnitude,growth
0,Biosciences,85,0.032,17.0,177.272727
1,Child care & preschool,173,0.066,34.6,-23.622047
2,Development & learning,260,0.099,52.0,0.714286
4,Health,920,0.351,184.0,5.482042
5,Parenting,261,0.1,52.2,-28.491620
6,Society,2,0.001,0.4,inf


Trends information for the heat map

In [43]:
chart_trends.estimate_trend_type(
    patent_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)

,type,counts,counts_prop,magnitude,growth,trend_type_suggestion
0,Biosciences,85,0.032,17.0,177.272727,emerging
1,Child care & preschool,173,0.066,34.6,-23.622047,dormant
2,Development & learning,260,0.099,52.0,0.714286,hot
4,Health,920,0.351,184.0,5.482042,hot
5,Parenting,261,0.1,52.2,-28.491620,stable
6,Society,2,0.001,0.4,inf,emerging


- Society has such a tiny number of patents we'll designate as 'dormant' instead
- Development and learning had a growth rate very close to 0, so not that strongly emerging - more between dormant and emerging.

In [44]:
fig = trends_chart(patent_application_trends)
fig

alt.Chart(...)

### Geographical distribution

In [45]:
_patents_countries_df = (
    patents_exploded_df
    .query("year <= 2023")
    .query('subtype in @tech_subtypes')
)

n_total_with_codes = len(
    _patents_countries_df
    .drop_duplicates('id')
    .query("year >= 2019")
    .query("year <= 2023")
    .dropna(subset=['country_code'])
)

patents_countries_df, _ = utils.get_geographical_distribution(_patents_countries_df)
patents_countries_df = (
    patents_countries_df
    .assign(proportion = lambda df: df.magnitude*5 / n_total_with_codes)
    .sort_values('proportion', ascending=False)
)
patents_countries_df.head(15)

,magnitude,growth,country_code,proportion
0,334.6,-15.964126,CN,0.639037
1,80.0,32.596685,KR,0.152788
2,36.6,-13.274336,US,0.069901
6,20.8,-9.836066,WO,0.039725
11,12.2,11.428571,JP,0.023300
4,7.8,141.666667,EP,0.014897
5,6.4,69.230769,TW,0.012223
3,4.2,800.000000,AU,0.008021
9,2.8,120.000000,TR,0.005348
7,2.8,125.000000,CA,0.005348


In [46]:
openalex_US = patents_countries_df.query("country_code=='CN'").proportion.iloc[0]
logging.info(f'CN proportion: {openalex_US:.2f}')

openalex_US = patents_countries_df.query("country_code=='US'").proportion.iloc[0]
logging.info(f'US proportion: {openalex_US:.2f}')

openalex_GB = patents_countries_df.query("country_code=='GB'").proportion.iloc[0]
logging.info(f'GB proportion: {openalex_GB:.2f}')

2024-08-06 11:25:51,022 - root - INFO - CN proportion: 0.64
2024-08-06 11:25:51,023 - root - INFO - US proportion: 0.07
2024-08-06 11:25:51,024 - root - INFO - GB proportion: 0.00


### Baseline growth across all sectors 

In [47]:
# Load the data
patents_baseline_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/baseline_data_patents.csv',
        download_as='dataframe'
    )
)

In [48]:
trends_baseline = au.ts_magnitude_growth_(
    ts_df = patents_baseline_df,
    year_start = 2019,
    year_end = 2023  
)
patents_baseline_magnitude = trends_baseline.loc['counts'].magnitude
patents_baseline_growth = trends_baseline.loc['counts'].growth
logging.info(f"UKRI baseline growth: {patents_baseline_growth:.2f}%")

2024-08-06 11:25:51,122 - root - INFO - UKRI baseline growth: 31.19%


## Venture funding

- Fig 2: Growth and total early-years digital tech funding in 2019-2023
- Fig 3: Venture funding by deal size over years
- Fig 4: Proportion of early-years funding associated with digital technologies
- Fig 5: Proportion and growth of major digital tech funding in 2019-2023
- Fig 6: Proportion and growth of major application areas in 2019-2023
- Baseline funding growth across all sectors in 2019-2023
- Geographical distribution


In [49]:
# Load the data
crunchbase_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/data_crunchbase_2024.csv',
        download_as='dataframe'
    )
    # Remove the instances tagged with expressive arts due to too much noise
    .query("topics != 'arts'")
)

# Explode by topics
crunchbase_exploded_df = utils.explode_data(crunchbase_df, is_crunchbase=True).query("topics != 'arts'")

# Get relevant document ids
crunchbase_tech_ids = get_tech_ids(crunchbase_exploded_df)
crunchbase_tech_ids_5y = get_tech_ids_5y(crunchbase_exploded_df)


In [50]:
logging.info(f"Total number of venture funding rounds {len(crunchbase_df)}")

2024-08-06 11:25:51,341 - root - INFO - Total number of venture funding rounds 2571


### Growth and total early-years digital tech investment

Figure 2

In [51]:
# Filter only technology projects
crunchbase_tech_type_df = (
    crunchbase_exploded_df
    .query("id in @crunchbase_tech_ids")
    .drop_duplicates(['id'])
)

crunchbase_ts_amounts_tech = utils.get_timeseries(crunchbase_tech_type_df, column='amount')
crunchbase_ts_counts_tech = utils.get_timeseries(crunchbase_tech_type_df, column='id')
utils.plot_quick_ts(crunchbase_ts_amounts_tech, 'amount')

alt.Chart(...)

In [52]:
# Get magnitude and growth
magnitude_growth_crunchbase = au.ts_magnitude_growth_(
    crunchbase_ts_amounts_tech,
    year_start = 2019,
    year_end = 2023
)

growth, magnitude = report_magnitude_growth(magnitude_growth_crunchbase)

2024-08-06 11:25:51,367 - root - INFO - Growth in 2019-2023: 41.20%
2024-08-06 11:25:51,367 - root - INFO - Total in 2019-2023: 4232509.55


Check growth if removing large deals (larger than £100M)

In [53]:
# Filter only technology projects
crunchbase_tech_type_df_wout_large = (
    crunchbase_exploded_df
    .query("amount < 100000")
    .query("id in @crunchbase_tech_ids")
    .drop_duplicates(['id'])
)

ts_amounts_tech = utils.get_timeseries(crunchbase_tech_type_df_wout_large, column='amount')
ts_counts_tech = utils.get_timeseries(crunchbase_tech_type_df_wout_large, column='id')
utils.plot_quick_ts(ts_amounts_tech, 'amount')

alt.Chart(...)

In [54]:
# Get magnitude and growth
magnitude_growth_crunchbase = au.ts_magnitude_growth_(
    ts_amounts_tech,
    year_start = 2019,
    year_end = 2023
)

growth, magnitude = report_magnitude_growth(magnitude_growth_crunchbase)

2024-08-06 11:25:51,396 - root - INFO - Growth in 2019-2023: 3.26%
2024-08-06 11:25:51,397 - root - INFO - Total in 2019-2023: 2591931.87


### Venture funding by deal size over years

Figure 3

In [55]:
deal_order = ["n/a", "£0-5M", "£5-20M", "£20-100M", "£100M+"]

def deal_amount_to_range_coarse(
    amount: float, currency: str = "£", categories: bool = True
) -> str:
    """
    Convert amounts to range in millions
    Args:
        amount: Investment amount (in GBP thousands)
        categories: If True, adding indicative deal categories
        currency: Currency symbol
    """
    amount /= 1e3
    if (amount >= 0.001) and (amount <= 5):
        return f"{currency}0-5M" if not categories else f"{currency}0-5M"
    elif (amount > 5) and (amount <= 20):
        return f"{currency}5-20M" if not categories else f"{currency}5-20M"
    elif (amount > 20) and (amount <= 100):
        return f"{currency}20-100M" if not categories else f"{currency}20-100M"
    elif amount > 100:
        return f"{currency}100M+"
    else:
        return "n/a"

In [56]:
funding_df_ranges = (
    crunchbase_df
    .query("id in @crunchbase_tech_ids")
    .assign(_amount = lambda df: df.amount/1000)
    .assign(deal_type=lambda df: df.amount.apply(deal_amount_to_range_coarse))
    .astype({"deal_type": "category"})
    .assign(deal_type=lambda x: x.deal_type.cat.set_categories(deal_order))
    .drop_duplicates('id')
)

deal_data = (
    funding_df_ranges.groupby(["year", "deal_type"], as_index=True)
    .agg(
        counts=("id", "count"),
        total_amount=("amount", "sum"),
    )
    .reset_index()
    .query("year >= 2013")
    .query("year <= 2023")
    .assign(total_amount=lambda df: df.total_amount / 1000)
)
deal_data_wide_df = (
    deal_data.pivot(index="year", columns="deal_type", values="total_amount")
    .fillna(0)
    .astype(int)
    .reset_index()
)

deal_data_wide_df = (
    deal_data_wide_df
    .merge(deal_data.groupby('year').total_amount.sum().reset_index(), on='year', how='left')
)

deal_data_wide_df

,year,n/a,£0-5M,£5-20M,£20-100M,£100M+,total_amount
0,2013,0,55,46,22,0,124.429639
1,2014,0,80,88,174,0,343.423021
2,2015,0,95,89,424,0,609.399647
3,2016,0,101,158,307,642,1209.904105
4,2017,0,109,179,40,0,330.011927
5,2018,0,121,201,218,114,656.101867
6,2019,0,109,156,236,348,851.677676
7,2020,0,90,123,456,115,785.950728
8,2021,0,110,170,564,1175,2021.634749
9,2022,0,85,163,21,0,269.569469


In [57]:
deal_data_wide_counts_df = (
    deal_data.pivot(index="year", columns="deal_type", values="counts")
    .fillna(0)
    .astype(int)
    .reset_index()
)
deal_data_wide_counts_df

deal_type,year,n/a,£0-5M,£5-20M,£20-100M,£100M+
0,2013,0,102,5,1,0
1,2014,0,120,9,5,0
2,2015,0,126,9,8,0
3,2016,0,121,16,7,3
4,2017,0,126,18,2,0
5,2018,0,114,18,6,1
6,2019,0,104,18,6,3
7,2020,0,103,13,8,1
8,2021,1,100,17,13,5
9,2022,0,62,14,1,0


In [58]:
# Sense check
deal_data_wide_df.query("year >= 2019 and year <= 2023").total_amount.sum()

4232.509547607872

### Proportion of early-years investment associated with digital technologies

Figure 4

In [59]:
# Check Technology proportion using another approach
crunchbase_category_distribution_df = utils.get_data_distribution(
    crunchbase_exploded_df.query("year >= 2019 and year <= 2023"),
    column='type', 
    values=['id', 'amount']
)

# Get the proportion of technology projects
crunchbase_prop = crunchbase_category_distribution_df.query("type == @TECH").amount_prop.iloc[0]
logging.info(f"Proportion of investment: {crunchbase_prop:.2f}")

# See all major categories
crunchbase_category_distribution_df

2024-08-06 11:25:51,452 - root - INFO - Proportion of investment: 0.56


,type,counts,counts_prop,amount,amount_prop
0,Biosciences,23,0.021,55364.986316,0.007
1,Child care & preschool,127,0.118,610225.151123,0.081
2,Development & learning,232,0.215,2292446.085317,0.304
3,General,420,0.39,2695685.632986,0.357
4,Health,425,0.394,3026034.566599,0.401
5,Parenting,177,0.164,720381.014435,0.095
6,Society,108,0.1,509445.801672,0.067
7,Technology,513,0.476,4232509.547608,0.561


### Digital tech trends in 2019-2023 (investment)

Figure 5

In [60]:
crunchbase_tech_trends = get_tech_trends(
    crunchbase_exploded_df.query("year <= 2023"), 
    crunchbase_tech_ids, 
    crunchbase_tech_ids_5y, 
    ['amount', 'id']
)

crunchbase_tech_trends

,subtype,amount,amount_prop,counts,counts_prop,type,magnitude,growth
0,AI,625016.449792,0.148,173,0.337,Technology,125003.289958,73.901417
1,Immersive tech,284661.474141,0.067,62,0.121,Technology,56932.294828,-17.969141
2,Internet,2892628.722755,0.683,110,0.214,Technology,578525.744551,41.941131
3,Mobile,1377192.183934,0.325,234,0.456,Technology,275438.436787,13.107150
4,Operations,435877.562785,0.103,89,0.173,Child care & preschool,87175.512557,8.858934


### Application area trends in 2019-2023 (investment)

Figure 6

In [61]:
crunchbase_application_trends = get_application_trends(
    crunchbase_exploded_df.query("year <= 2023"), 
    crunchbase_tech_ids, 
    crunchbase_tech_ids_5y, 
    ['amount', 'id']
)

crunchbase_application_trends


,type,amount,amount_prop,counts,counts_prop,magnitude,growth
0,Biosciences,9026.586836,0.002,9,0.018,1805.317367,1748.296224
1,Child care & preschool,463498.943387,0.11,102,0.199,92699.788677,-0.863038
2,Development & learning,1459553.275091,0.345,147,0.287,291910.655018,44.765309
4,Health,620837.404541,0.147,144,0.281,124167.480908,40.466729
5,Parenting,273485.631675,0.065,105,0.205,54697.126335,-7.308249
6,Society,165733.006945,0.039,50,0.097,33146.601389,22.534738


Trends information for the heat map

In [62]:
chart_trends.estimate_trend_type(
    crunchbase_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)[['type', 'magnitude', 'growth', 'trend_type_suggestion']]

,type,magnitude,growth,trend_type_suggestion
0,Biosciences,1805.317367,1748.296224,emerging
1,Child care & preschool,92699.788677,-0.863038,stable
2,Development & learning,291910.655018,44.765309,hot
4,Health,124167.480908,40.466729,hot
5,Parenting,54697.126335,-7.308249,dormant
6,Society,33146.601389,22.534738,emerging


In [63]:
fig = trends_chart(
    crunchbase_application_trends
    .query("type != 'Biosciences'")
)
fig

alt.Chart(...)

### Geographical distribution

In [64]:
_crunchbase_countries_df = (
    crunchbase_exploded_df
    .query("year <= 2023")
    .query('id in @crunchbase_tech_ids')
)

n_total_with_codes = (
    _crunchbase_countries_df
    .query("year >= 2019")
    .query("year <= 2023")
    .drop_duplicates('id')
    .dropna(subset=['country_code'])
    .amount.sum()
)

crunchbase_countries_df, _ = utils.get_geographical_distribution(
    _crunchbase_countries_df,
    column = 'amount',
)
crunchbase_countries_df = (
    crunchbase_countries_df
    .assign(total = lambda df: df.magnitude*5)
    .assign(proportion = lambda df: df.total / n_total_with_codes)
    .sort_values('proportion', ascending=False)
)
crunchbase_countries_df.head(10)

,magnitude,growth,country_code,total,proportion
3,523630.007604,147.721838,USA,2.618150e+06,0.618581
11,158982.268980,13.455687,IND,7.949113e+05,0.187811
1,46772.956806,-81.965750,CHN,2.338648e+05,0.055254
8,37109.720306,-76.741053,GBR,1.855486e+05,0.043839
4,11868.538098,602.395266,CAN,5.934269e+04,0.014021
31,10437.474855,-23.922810,JPN,5.218737e+04,0.012330
35,9495.637346,1626.850918,KOR,4.747819e+04,0.011218
6,7872.595920,179.978612,ESP,3.936298e+04,0.009300
5,7784.813000,135.579853,DEU,3.892407e+04,0.009196
0,6010.114111,235.287954,FRA,3.005057e+04,0.007100


In [65]:
# Double check GBR total investment
(
    crunchbase_exploded_df
    .query("id in @crunchbase_tech_ids_5y")
    .query("year <= 2023")
    .query("country_code == 'GBR'")
    .drop_duplicates('id')
    .amount.sum()
)

185548.6015294242

In [66]:
# Double check investment types
sorted(list(crunchbase_exploded_df.investment_type.unique()))

['angel',
 'convertible_note',
 'equity_crowdfunding',
 'non_equity_assistance',
 'pre_seed',
 'product_crowdfunding',
 'secondary_market',
 'seed',
 'series_a',
 'series_b',
 'series_c',
 'series_d',
 'series_e',
 'series_unknown']

### Baseline growth across all sectors (venture funding)

In [67]:
# Load the data
crunchbase_baseline_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/baseline_data_crunchbase.csv',
        download_as='dataframe'
    )
)

In [68]:
trends_baseline = au.ts_magnitude_growth_(
    ts_df = crunchbase_baseline_df,
    year_start = 2019,
    year_end = 2023  
)
crunchbase_baseline_magnitude = trends_baseline.loc['amount'].magnitude
crunchbase_baseline_growth = trends_baseline.loc['amount'].growth
logging.info(f"Investment baseline growth: {crunchbase_baseline_growth:.2f}%")

2024-08-06 11:25:52,034 - root - INFO - Investment baseline growth: 81.75%


### Baseline growth across all sectors (edtech)

In [69]:
# Load the data
crunchbase_baseline_df_edtech = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/baseline_data_crunchbase_edtech.csv',
        download_as='dataframe'
    )
)

trends_baseline = au.ts_magnitude_growth_(
    ts_df = crunchbase_baseline_df_edtech,
    year_start = 2019,
    year_end = 2023  
)
crunchbase_baseline_magnitude_edtech = trends_baseline.loc['amount'].magnitude*5
crunchbase_baseline_growth_edtech = trends_baseline.loc['amount'].growth
logging.info(f"Edtech investment baseline growth: {crunchbase_baseline_growth_edtech:.2f}%")
logging.info(f"Edtech magnitude: {crunchbase_baseline_magnitude_edtech:.2f}")

2024-08-06 11:14:49,128 - root - INFO - Edtech investment baseline growth: 34.44%
2024-08-06 11:14:49,129 - root - INFO - Edtech magnitude: 32859079.79


# Technical appendix

In [69]:
FIGURE_DIR = PROJECT_DIR / 'outputs/figures/final'

CATS = [
    'Health',
    'Development & learning',
    'Child care & preschool',
    'Parenting',
    'Society',
    'Biosciences',
]

TS_FIG_HEIGHT = 150
TS_FIG_WIDTH = 250
DEF_COLOUR = pu.NESTA_COLOURS[0]
SECONDARY_COLOUR = pu.NESTA_COLOURS[1]

## Detailed time series for Figure 2

### Data used for growth estimates

In [70]:
def simple_bar_chart(
        data,
        y_field,
        title,
        denominator=1,
        colour=DEF_COLOUR,
        round_int=1,):
    fig = (alt.Chart(
            (
                data
                .assign(**{y_field: lambda df: df[y_field] / denominator})
            ),
            width = 350,
            height = 150,
        ).encode(
            x=alt.X('year:O', title=''),
            y=alt.Y(f'{y_field}:Q', title=''),
            color=alt.value((pu.NESTA_COLOURS[0])),
            tooltip=['year', y_field],
        )
    )
    
    fig_text = (
        fig
        .encode(
            text=alt.Text(f'{y_field}:Q', format=f'.{round_int}f')
        )
    )
    fig = fig.mark_bar().properties(title=title) + fig_text.mark_text(dy=-5)
    return (
        pu.configure_plots(fig, title)
        .configure_legend(title=None)
    )
    # return fig


In [72]:
fig = simple_bar_chart(
    ukri_ts_amounts_tech,
    'amount',
    'Research funding (£ millions)',
    1e+3)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_ukri_digital_tech.png', scale_factor=2.0)

alt.LayerChart(...)

In [73]:
fig = simple_bar_chart(
    openalex_ts_counts_tech.query("year >= 2017"),
    'counts',
    'Research publication counts',
    1,
    round_int=0)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_openalex_digital_tech.png', scale_factor=2.0)

alt.LayerChart(...)

In [76]:
fig = simple_bar_chart(
    patents_ts_counts_tech,
    'counts',
    'Patent application counts',
    1,
    round_int=0)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_patents_digital_tech.png', scale_factor=2.0)

alt.LayerChart(...)

In [77]:
fig = simple_bar_chart(
    ts_counts_tech_wout_CN,
    'counts',
    'Patent application counts (excluding China)',
    1,
    round_int=0)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_patents_wout_china_digital_tech.png', scale_factor=2.0)

alt.LayerChart(...)

In [75]:
fig = simple_bar_chart(
    crunchbase_ts_amounts_tech,
    'amount',
    'Venture funding (£ millions)',
    1e+3)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_crunchbase_digital_tech.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Figure2_crunchbase_digital_tech.html')

alt.LayerChart(...)

In [78]:
fig = simple_bar_chart(
    crunchbase_ts_amounts_tech,
    'amount',
    'Venture funding (£ millions)',
    1e+3)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_crunchbase_digital_tech.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Figure2_crunchbase_digital_tech.html')

alt.LayerChart(...)

### Baselines

In [79]:
fig = simple_bar_chart(
    ukri_baseline_df,
    'amount',
    'Baseline research funding (£ billions)',
    1e+6,
    round_int=2)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_ukri_baseline.png', scale_factor=2.0)

alt.LayerChart(...)

In [80]:
fig = simple_bar_chart(
    openalex_baseline_df.query("year >= 2017"),
    'counts',
    'Baseline research publication counts (millions)',
    1e+6,
    round_int=2)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_openalex_baseline.png', scale_factor=2.0)

alt.LayerChart(...)

In [81]:
fig = simple_bar_chart(
    patents_baseline_df,
    'counts',
    'Baseline patent application counts (millions)',
    1e+6,
    round_int=2)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_patents_baseline.png', scale_factor=2.0)

alt.LayerChart(...)

In [82]:
fig = simple_bar_chart(
    crunchbase_baseline_df,
    'amount',
    'Baseline venture funding (£ billions)',
    1e+6)
fig.display()

fig.save(FIGURE_DIR / 'Figure2_crunchbase_baseline.png', scale_factor=2.0)

alt.LayerChart(...)

## Digital tech

### Detailed time series, all datasets

In [84]:
def create_bar_chart(data, dataset_name, y_field, y_title, category, colour=DEF_COLOUR):
    return (
        alt.Chart(
            (
                data
                .query(f"dataset == '{dataset_name}'")
                .query(f"type == '{category}'")
            ),
            height=TS_FIG_HEIGHT,
            width=TS_FIG_WIDTH,
        )
        .mark_bar()
        .encode(
            x=alt.X('year:O', title=''),
            y=alt.Y(f'{y_field}:Q', title=y_title),
            color=alt.value(colour),
            tooltip=['year', y_field],
        )
        .properties(
            title=f'{category} {dataset_name.lower()}'
        )
    )

def create_step_chart(data, dataset_name, y_field, y_title, category, colour=SECONDARY_COLOUR):
    return (
        alt.Chart(
            (
                data
                .query(f"dataset == '{dataset_name}'")
                .query(f"type == '{category}'")
            ),
            height=TS_FIG_HEIGHT,
            width=TS_FIG_WIDTH,
        )
        .mark_line(interpolate='step')
        .encode(
            x=alt.X('year:O', title=''),
            y=alt.Y(f'{y_field}:Q', title=y_title),
            color=alt.value(colour),
            tooltip=['year', y_field],
        )
    )

def category_ts_figs(category_ts, category):
    ukri_step_fig = create_step_chart(
        category_ts, 
        'Research funding', 
        'counts', 
        'Project counts', 
        category
    )

    ukri_funds_fig = create_bar_chart(
        category_ts, 
        'Research funding', 
        'amount', 
        'Research funding (£ millions)', 
        category
    )

    ukri_ts_fig = alt.layer(ukri_funds_fig, ukri_step_fig).resolve_scale(y='independent')

    openalex_ts_fig = create_bar_chart(
        category_ts, 
        'Publications', 
        'counts', 
        'Publication counts', 
        category
    )

    patents_ts_fig = create_bar_chart(
        category_ts, 
        'Patents', 
        'counts', 
        'Patent counts', 
        category
    )

    crunchbase_step_fig = create_step_chart(
        category_ts, 
        'Venture funding', 
        'counts', 
        'Funding round counts', 
        category
    )

    crunchbase_funds_fig = create_bar_chart(
        category_ts, 
        'Venture funding', 
        'amount', 
        'Venture funding (£ millions)', 
        category
    )

    crunchbase_ts_fig = alt.layer(crunchbase_funds_fig, crunchbase_step_fig).resolve_scale(y='independent')

    upstream_combined_fig = alt.hconcat(
        ukri_ts_fig,
        openalex_ts_fig,
        spacing = 20,
    )

    downstream_combined_fig = alt.hconcat(
        patents_ts_fig,
        crunchbase_ts_fig,
        spacing = 60,
    )

    combined_fig = alt.vconcat(
        upstream_combined_fig,
        downstream_combined_fig,
        spacing = 30,
    )

    return pu.configure_plots(combined_fig)

In [85]:
ukri_tech_ts = (
    get_tech_ts(
        ukri_exploded_df,
        ukri_tech_ids,
        ['id', 'amount'],
    )
    .assign(dataset = 'Research funding')
)

openalex_tech_ts = (
    get_tech_ts(
        openalex_exploded_df,
        openalex_tech_ids,
        ['id'],
    )
    .assign(dataset = 'Publications')
)
# Replace counts values with 0 for all columns 'years' betwene 2013 and 2016
openalex_tech_ts.loc[
    openalex_tech_ts.year < 2017, 
    openalex_tech_ts.columns.str.contains('counts')
] = 0

patents_tech_ts = (
    get_tech_ts(
        patents_exploded_df,
        patents_tech_ids,
        ['id'],
    )
    .assign(dataset = 'Patents')
)

crunchbase_tech_ts = (
    get_tech_ts(
        crunchbase_exploded_df,
        crunchbase_tech_ids,
        ['id', 'amount'],
    )
    .assign(dataset = 'Venture funding')
)

tech_ts_df = (
    pd.concat([ukri_tech_ts, openalex_tech_ts, patents_tech_ts, crunchbase_tech_ts])
    .assign(amount = lambda df: df.amount/1e3)
    .drop('type', axis=1)
    .rename(columns = {'subtype': 'type'})
)


In [86]:
for cat in ['AI', 'Mobile', 'Internet', 'Immersive tech']:
    fig = category_ts_figs(tech_ts_df.query("year <= 2023"), category=cat)
    # save as png with dpi=300
    fig.save(FIGURE_DIR / f'Digital_tech_{cat}_ts.png', scale_factor=2.0)
    # save as html
    fig.save(FIGURE_DIR / f'Digital_tech_{cat}_ts.html')


## Application areas

### Detailed time series, all datasets

In [87]:
# Prepare time series data for all applications
ukri_applications_ts = (
    utils.get_data_distribution(
        ukri_exploded_df.query('id in @ukri_tech_ids'),
        column='type', 
        values=['id', 'amount'],
        ts=True
    )
    .assign(dataset = 'Research funding')
)

openalex_applications_ts = (
    utils.get_data_distribution(
        openalex_exploded_df.query('id in @openalex_tech_ids'),
        column='type', 
        values=['id'],
        ts=True
    )
    .assign(dataset = 'Publications')
)
# Replace counts values with 0 for all columns 'years' betwene 2013 and 2016
openalex_applications_ts.loc[
    openalex_applications_ts.year < 2017, 
    openalex_applications_ts.columns.str.contains('counts')
] = 0

patents_applications_ts = (
    utils.get_data_distribution(
        patents_exploded_df.query('id in @patents_tech_ids'),
        column='type', 
        values=['id'],
        ts=True
    )
    .assign(dataset = 'Patents')
)

crunchbase_applications_ts = (
    utils.get_data_distribution(
        crunchbase_exploded_df.query('id in @crunchbase_tech_ids'),
        column='type', 
        values=['id', 'amount'],
        ts=True
    )
    .assign(dataset = 'Venture funding')
)

applications_ts = (
    pd.concat([
        ukri_applications_ts,
        openalex_applications_ts,
        patents_applications_ts,
        crunchbase_applications_ts
    ], ignore_index=True)
    # Convert to millions
    .assign(amount = lambda df: df.amount / 1000)
)

In [90]:
for cat in CATS:
    fig = category_ts_figs(applications_ts.query("year <= 2023"), cat)
    # save as png with dpi=300
    fig.save(FIGURE_DIR / f'Application_areas_{cat}_ts.png', scale_factor=2.0)
    # save as html
    fig.save(FIGURE_DIR / f'Application_areas_{cat}_ts.html')
    

### Applications: detailed breakdowns of major categories

In [88]:
CATS_COLOURS = {
    'Health': pu.NESTA_COLOURS[0],
    'Development & learning': pu.NESTA_COLOURS[1],
    'Child care & preschool': pu.NESTA_COLOURS[4],
    'Parenting': pu.NESTA_COLOURS[9],
    'Society': pu.NESTA_COLOURS[2],
    'Biosciences': pu.NESTA_COLOURS[5],
}

In [91]:
applications_trends = (
    pd.concat([
        ukri_application_trends.assign(dataset='Research funding'),
        openalex_application_trends.assign(dataset='Publications'),
        patent_application_trends.assign(dataset='Patents'),
        crunchbase_application_trends.assign(dataset='Venture funding'),
    ], ignore_index=True)
    .assign(amount = lambda df: df.amount / 1000)
)

In [92]:
# bar chart
def create_application_bar_chart(
        data, 
        dataset_name, 
        title, 
        values, 
        show_yaxis=True, 
        category='type', 
        fig_height=150,
        sort_order = CATS,
    ):
    fig = (
        alt.Chart(
            data.query(f"dataset == '{dataset_name}'"),
            height=fig_height,
            width=100,
        )
        .encode(
            x=alt.X(f'{values}:Q', title=title),
            y=alt.Y(f'{category}:N', title='', sort=sort_order, axis=alt.Axis(labels=False) if not show_yaxis else alt.Axis()),
            color=alt.Color(
                f'type:N', 
                scale=alt.Scale(domain=list(CATS_COLOURS.keys()), range=list(CATS_COLOURS.values())),
                legend=None,
            ),
            tooltip=['dataset', 'type', values, 'growth'],
            text=alt.Text(f'{values}:Q', format=".0f")
        )
        .properties(
            title=dataset_name
        )
    )
    return fig.mark_bar() + fig.mark_text(align='left', dx=2)

def application_bar_charts(applications_trends, category, fig_height=150, sort_order=CATS):
    fig = alt.hconcat(
        create_application_bar_chart(
            applications_trends,
            'Research funding',
            'Research funding (£ millions)',
            'amount',
            show_yaxis=True,
            category=category,
            fig_height=fig_height,
            sort_order=sort_order,
        ),
        create_application_bar_chart(
            applications_trends,
            'Publications',
            'Publication counts',
            'counts',
            show_yaxis=False,
            category=category, 
            fig_height=fig_height,
            sort_order=sort_order,                   
        ),
        create_application_bar_chart(
            applications_trends,
            'Patents',
            'Patent counts',
            'counts',
            show_yaxis=False,
            category=category,  
            fig_height=fig_height,      
            sort_order=sort_order,            
        ),    
        create_application_bar_chart(
            applications_trends,
            'Venture funding',
            'Investment (£ millions)',
            'amount',
            show_yaxis=False,
            category=category,   
            fig_height=fig_height,  
            sort_order=sort_order,               
        )
    ).configure_title(anchor='middle')
    return pu.configure_plots(fig)

fig = application_bar_charts(applications_trends, 'type')
fig

alt.HConcatChart(...)

In [93]:
fig.save(FIGURE_DIR / 'Applications_bar_charts_major.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_bar_charts_major.html')

### Applications: detailed breakdowns of minor categories

In [94]:
def add_missing_subtypes(trends_minor, topics_df, dataset_name):
    missing_subtypes = (
        set(topics_df.subtype.unique()) - set(trends_minor.subtype.unique())
    )
    return (
        pd.concat([
            trends_minor,
            pd.DataFrame(
                {
                    'subtype': list(missing_subtypes),
                    'amount': 0,
                    'counts': 0,
                    'dataset': dataset_name,
                }
            )
        ])
    )

def process_minor_application_trends_df(trends_minor, dataset_name):
    return (
        trends_minor
        .dropna(subset=['type'])
        .assign(dataset = dataset_name)
        .pipe(add_missing_subtypes, topics_df, dataset_name)
        .drop(columns=['type', 'type_'])
        .merge(topics_df[['subtype', 'type']], on='subtype', how='left')        
    )

In [95]:
ukri_application_trends_minor = process_minor_application_trends_df(
    get_application_trends(
        ukri_exploded_df.query("year <= 2023"), 
        ukri_tech_ids, 
        ukri_tech_ids_5y, 
        ['amount', 'id'],
        column = 'subtype',
    ),
    dataset_name = 'Research funding'
)

openalex_application_trends_minor = process_minor_application_trends_df(
    get_application_trends(
        openalex_exploded_df.query("year <= 2023"), 
        openalex_tech_ids, 
        openalex_tech_ids_5y, 
        ['id'],
        column = 'subtype',
    ),
    dataset_name = 'Publications'
)

patents_application_trends_minor = process_minor_application_trends_df(
    get_application_trends(
        patents_exploded_df.query("year <= 2023"), 
        patents_tech_ids, 
        patents_tech_ids_5y, 
        ['id'],
        column = 'subtype',
    ),
    dataset_name = 'Patents'
)

crunchbase_application_trends_minor = process_minor_application_trends_df(
    get_application_trends(
        crunchbase_exploded_df.query("year <= 2023"), 
        crunchbase_tech_ids, 
        crunchbase_tech_ids_5y, 
        ['amount', 'id'],
        column = 'subtype',
    ),
    dataset_name = 'Venture funding'
)

applications_trends_minor = (
    pd.concat([
        ukri_application_trends_minor,
        openalex_application_trends_minor,
        patents_application_trends_minor,
        crunchbase_application_trends_minor,
    ], ignore_index=True)
    .assign(amount = lambda df: df.amount / 1000)
)

sort_order_df = (
    applications_trends_minor
    .groupby(['type', 'subtype'])
    .agg(total=('counts', 'sum'))
    .reset_index()
    .assign(type = lambda df: df.type.astype('category').cat.set_categories(CATS))
    .sort_values(['type', 'total'], ascending=[True, False])
    .dropna(subset=['type'])
)

In [97]:
applications_trends_minor.head(1)

,subtype,amount,amount_prop,counts,counts_prop,magnitude,growth,dataset,type
0,AI,40.910811,0.695,62,0.59,8182.1622,147.968645,Research funding,Technology


In [107]:
fig = (
    application_bar_charts(
        (
            applications_trends_minor
            .query("subtype in @sort_order_df.subtype")
            .query("subtype != 'Expressive arts and design'")
        ),
        'subtype',
        fig_height=400,
        sort_order = sort_order_df.subtype.to_list()
    )
    .configure_axisY(grid=True)
    .configure_axisX(grid=False)
)
fig.display()
fig.save(FIGURE_DIR / 'Applications_bar_charts_minor.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_bar_charts_minor.html')

alt.HConcatChart(...)

### Applications x Digital tech: detailed breakdowns of major categories


In [112]:
def applications_x_digital_tech_df_type(
        data_exploded_df, 
        applications_trends, 
        tech_ids,
        tech_ids_5y,
        values
    ):
    # Empty dataframe with all subtypes
    _df = topics_df.query("type in @CATS").query("topic != 'arts'")[['type']].drop_duplicates()
    _values = 'id' if values == 'counts' else values
    # Go through each tech category
    for tech_topic in ['AI', 'Internet', 'Mobile', 'Immersive tech']:
        # Select relevant technology type
        selected_ids = data_exploded_df.query("subtype == @tech_topic").id.to_list()
        # Get application trends for this subset
        counts_df = get_application_trends(
            data_exploded_df.query("year <= 2023").query("id in @selected_ids"), 
            tech_ids, 
            tech_ids_5y, 
            [_values],
            column = 'type',
        )[[values, 'type']].rename(columns={values: tech_topic})
        # Add to the final dataframe
        _df = _df.merge(counts_df, on='type', how='left')
    _df = _df.fillna(0)

    final_df = (
        applications_trends[['type', values]]
        .merge(_df, on='type')
        .rename(columns={values: 'Total'})
        .assign(subtype = lambda df: df.type.astype('category').cat.set_categories(CATS))
        .sort_values('type')
    )[['type', 'AI', 'Mobile', 'Internet', 'Immersive tech', 'Total']]

    return final_df

In [113]:
ukri_x_digital_tech_major = applications_x_digital_tech_df_type(
    ukri_exploded_df.assign(amount = lambda df: df.amount/1000).query("year <= 2023"), 
    applications_trends.query("dataset == 'Research funding'"), 
    ukri_tech_ids,
    ukri_tech_ids_5y,
    'amount'
)

openalex_x_digital_tech_major = applications_x_digital_tech_df_type(
    openalex_exploded_df.query("year <= 2023"), 
    applications_trends.query("dataset == 'Publications'"), 
    openalex_tech_ids,
    openalex_tech_ids_5y,
    'counts'
)

patents_x_digital_tech_major = applications_x_digital_tech_df_type(
    patents_exploded_df.query("year <= 2023"), 
    applications_trends.query("dataset == 'Patents'"), 
    patents_tech_ids,
    patents_tech_ids_5y,
    'counts'
)

crunchbase_x_digital_tech_major = applications_x_digital_tech_df_type(
    crunchbase_exploded_df.assign(amount = lambda df: df.amount/1000).query("year <= 2023"), 
    applications_trends.query("dataset == 'Venture funding'"), 
    crunchbase_tech_ids,
    crunchbase_tech_ids_5y,
    'amount'
)

ukri_x_digital_tech_major_counts = applications_x_digital_tech_df_type(
    ukri_exploded_df.query("year <= 2023"), 
    applications_trends.query("dataset == 'Research funding'"), 
    ukri_tech_ids,
    ukri_tech_ids_5y,
    'counts'
)

crunchbase_x_digital_tech_major_counts = applications_x_digital_tech_df_type(
    crunchbase_exploded_df.query("year <= 2023"), 
    applications_trends.query("dataset == 'Venture funding'"), 
    crunchbase_tech_ids,
    crunchbase_tech_ids_5y,
    'counts'
)


In [114]:
def applications_x_digital_tech_chart_major(df, dataset_name, values='amount'):
    # transform to long format
    df_long = df.melt(id_vars=['type'], var_name='column', value_name=values)
    # altair facet grid with one column per digital tech area + total
    fig = (
        alt.Chart(
            width = 120,
        )
        .mark_bar()
        .encode(
            x=alt.X(f'{values}:Q', title=''),
            y=alt.Y('type:N', title='', sort=sort_order_df.subtype.to_list()),        
            # use the scale defined above
            color=alt.Color('type:N', scale=alt.Scale(domain=CATS, range=list(CATS_COLOURS.values())), legend=None),
            tooltip=['type', values]
        )
    )

    text = fig.mark_text(align='left', dx=2).encode(text=alt.Text(f'{values}:Q', format=".0f"))
    # adjust grid
    fig = (
        alt.layer(fig, text, data=df_long)
        .facet(
            column=alt.Column('column:N', title='', sort=['AI', 'Mobile', 'Internet', 'Immersive tech', 'Total'])
        ) 
    )       
    fig = (
        fig
        .configure_axisY(grid=True)
        .configure_axisX(grid=False)
    )
    fig = pu.configure_plots(fig, chart_title = dataset_name)
    return fig

In [117]:
applications_x_digital_tech_chart_major(
    ukri_x_digital_tech_major, 
    'Research funding (£ millions)',
    'amount'
)

alt.FacetChart(...)

In [111]:
fig = applications_x_digital_tech_chart_major(
    ukri_x_digital_tech_major, 
    'Research funding (£ millions)',
    'amount'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_ukri.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_ukri.html')

fig = applications_x_digital_tech_chart_major(
    openalex_x_digital_tech_major, 
    'Publications',
    'counts'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_openalex.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_openalex.html')

fig = applications_x_digital_tech_chart_major(
    patents_x_digital_tech_major, 
    'Patents',
    'counts'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_patents.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_patents.html')

fig = applications_x_digital_tech_chart_major(
    crunchbase_x_digital_tech_major, 
    'Venture funding (£ millions)',
    'amount'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_crunchbase.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_crunchbase.html')

fig = applications_x_digital_tech_chart_major(
    ukri_x_digital_tech_major_counts, 
    'Research projects counts',
    'counts'
)

fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_ukri_counts.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_ukri_counts.html')

fig = applications_x_digital_tech_chart_major(
    crunchbase_x_digital_tech_major_counts, 
    'Number of funding rounds',
    'counts'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_crunchbase_counts.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_major_crunchbase_counts.html')



### Applications x Digital tech: detailed breakdowns of minor categories

In [118]:
def applications_x_digital_tech_df(
        data_exploded_df, 
        application_trends_minor, 
        tech_ids,
        tech_ids_5y,
        values
    ):
    # Empty dataframe with all subtypes
    _df = topics_df.query("type in @CATS").query("topic != 'arts'")[['subtype']]
    _values = 'id' if values == 'counts' else values
    # Go through each tech category
    for tech_topic in ['AI', 'Internet', 'Mobile', 'Immersive tech']:
        # Select relevant technology type
        selected_ids = data_exploded_df.query("subtype == @tech_topic").id.to_list()
        # Get application trends for this subset
        counts_df = get_application_trends(
            data_exploded_df.query("year <= 2023").query("id in @selected_ids"), 
            tech_ids, 
            tech_ids_5y, 
            [_values],
            column = 'subtype',
        )[[values, 'subtype']].rename(columns={values: tech_topic})
        # Add to the final dataframe
        _df = _df.merge(counts_df, on='subtype', how='left')
    _df = _df.fillna(0)

    final_df = (
        application_trends_minor[['subtype', values]]
        .merge(_df, on='subtype')
        .rename(columns={values: 'Total'})
        .merge(topics_df[['subtype', 'type']], on='subtype', how='left')
        .assign(subtype = lambda df: df.subtype.astype('category').cat.set_categories(sort_order_df.subtype.to_list()))
        .sort_values('subtype')
    )[['type', 'subtype', 'AI', 'Mobile', 'Internet', 'Immersive tech', 'Total']]

    return final_df

In [119]:
ukri_x_digital_tech = applications_x_digital_tech_df(
    ukri_exploded_df.assign(amount = lambda df: df.amount/1000).query("year <= 2023"), 
    ukri_application_trends_minor.assign(amount = lambda df: df.amount/1000), 
    ukri_tech_ids,
    ukri_tech_ids_5y,
    'amount'
)

crunchbase_x_digital_tech = applications_x_digital_tech_df(
    crunchbase_exploded_df.assign(amount = lambda df: df.amount/1000).query("year <= 2023"), 
    crunchbase_application_trends_minor.assign(amount = lambda df: df.amount/1000), 
    crunchbase_tech_ids,
    crunchbase_tech_ids_5y,
    'amount'
)

patents_x_digital_tech = applications_x_digital_tech_df(
    patents_exploded_df.query("year <= 2023"), 
    patents_application_trends_minor, 
    patents_tech_ids,
    patents_tech_ids_5y,
    'counts'
)

openalex_x_digital_tech = applications_x_digital_tech_df(
    openalex_exploded_df.query("year <= 2023"), 
    openalex_application_trends_minor, 
    openalex_tech_ids,
    openalex_tech_ids_5y,
    'counts'
)

ukri_x_digital_tech.to_csv(FIGURE_DIR / 'Applications_x_digital_tech_ukri.csv', index=False)
crunchbase_x_digital_tech.to_csv(FIGURE_DIR / 'Applications_x_digital_tech_crunchbase.csv', index=False)
patents_x_digital_tech.to_csv(FIGURE_DIR / 'Applications_x_digital_tech_patents.csv', index=False)
openalex_x_digital_tech.to_csv(FIGURE_DIR / 'Applications_x_digital_tech_openalex.csv', index=False)

In [120]:
def applications_x_digital_tech_chart(df, dataset_name, values='amount'):
    # transform to long format
    df_long = df.melt(id_vars=['type', 'subtype'], var_name='column', value_name=values)
    # altair facet grid with one column per digital tech area + total
    fig = (
        alt.Chart(
            width = 120,
        )
        .mark_bar()
        .encode(
            x=alt.X(f'{values}:Q', title=''),
            y=alt.Y('subtype:N', title='', sort=sort_order_df.subtype.to_list()),        
            # use the scale defined above
            color=alt.Color('type:N', scale=alt.Scale(domain=CATS, range=list(CATS_COLOURS.values())), legend=None),
            tooltip=['subtype', values]
        )
    )

    text = fig.mark_text(align='left', dx=2).encode(text=alt.Text(f'{values}:Q', format=".0f"))
    # adjust grid
    fig = (
        alt.layer(fig, text, data=df_long)
        .facet(
            column=alt.Column('column:N', title='', sort=['AI', 'Mobile', 'Internet', 'Immersive tech', 'Total'])
        ) 
    )       
    fig = (
        fig
        .configure_axisY(grid=True)
        .configure_axisX(grid=False)
    )
    fig = pu.configure_plots(fig, chart_title = dataset_name)
    return fig

In [121]:
applications_x_digital_tech_chart(
    ukri_x_digital_tech, 
    'Research funding (£ millions)',
    'amount'
)

alt.FacetChart(...)

In [122]:
fig = applications_x_digital_tech_chart(
    ukri_x_digital_tech, 
    'Research funding (£ millions)',
    'amount'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_ukri.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_ukri.html')

fig = applications_x_digital_tech_chart(
    crunchbase_x_digital_tech, 
    'Venture funding (£ millions)',
    'amount'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_crunchbase.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_crunchbase.html')

fig = applications_x_digital_tech_chart(
    patents_x_digital_tech, 
    'Patent counts',
    'counts'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_patents.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_patents.html')

fig = applications_x_digital_tech_chart(
    openalex_x_digital_tech, 
    'Publication counts',
    'counts'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_openalex.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_openalex.html')



In [123]:
ukri_x_digital_tech_counts = applications_x_digital_tech_df(
    ukri_exploded_df.query("year <= 2023"), 
    ukri_application_trends_minor, 
    ukri_tech_ids,
    ukri_tech_ids_5y,
    'counts'
)

fig = applications_x_digital_tech_chart(
    ukri_x_digital_tech_counts, 
    'Research projects counts',
    'counts'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_ukri_counts.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_ukri_counts.html')



In [124]:
applications_x_digital_tech_chart(
    ukri_x_digital_tech_counts, 
    'Research projects counts',
    'counts'
)

alt.FacetChart(...)

In [450]:
crunchbase_x_digital_tech_counts = applications_x_digital_tech_df(
    crunchbase_exploded_df.query("year <= 2023"), 
    crunchbase_application_trends_minor, 
    crunchbase_tech_ids,
    crunchbase_tech_ids_5y,
    'counts'
)

fig = applications_x_digital_tech_chart(
    crunchbase_x_digital_tech_counts, 
    'Number of funding rounds',
    'counts'
)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_crunchbase_counts.png', scale_factor=2.0)
fig.save(FIGURE_DIR / 'Applications_x_digital_tech_crunchbase_counts.html')


## Growth and magnitude trends diagrams

In [913]:
chart_trends._epsilon = 0.05

In [125]:
ukri_trends_typology = chart_trends.estimate_trend_type(
    ukri_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)[['type', 'magnitude', 'growth', 'trend_type_suggestion']]
ukri_trends_typology

,type,magnitude,growth,trend_type_suggestion
0,Biosciences,2659.767,65.806908,hot*
1,Child care & preschool,518.043,299.955036,emerging
2,Development & learning,2528.182,-51.255006,dormant*
4,Health,8050.1558,210.796765,hot
5,Parenting,611.8658,2541.467658,emerging
6,Society,3251.175,148.675879,hot


In [126]:
chart_trends._epsilon = 0.025
mid_point = ukri_trends_typology.magnitude.median()

fig = chart_trends.mangitude_vs_growth_chart(
    data = (
        ukri_trends_typology
        .rename(columns = {'type': 'category'})
        .assign(growth = lambda df: df.growth / 100)
        # .assign(magnitude = lambda df: df.magnitude / 1000)
    ),
    x_limit=10000,
    y_limit= 3.5,
    mid_point=mid_point,
    baseline_growth=0,
    text_column = "category",
    values_label = "Average new funding per year (£ thousands)",
)
fig.display()
fig.save(FIGURE_DIR / 'Figure7_ukri_trends.png', scale_factor=2.0)

alt.LayerChart(...)

In [127]:
openalex_trends_typology = chart_trends.estimate_trend_type(
    openalex_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)[['type', 'magnitude', 'growth', 'trend_type_suggestion']]
openalex_trends_typology

,type,magnitude,growth,trend_type_suggestion
0,Biosciences,37.0,65.384615,emerging
1,Child care & preschool,79.6,96.078431,emerging*
2,Development & learning,144.6,90.405904,hot
4,Health,181.2,59.349593,hot
5,Parenting,20.4,97.297297,emerging
6,Society,85.8,70.930233,hot*


In [128]:
chart_trends._epsilon = 0.075
mid_point = openalex_trends_typology.magnitude.median()

fig = chart_trends.mangitude_vs_growth_chart(
    data = (
        openalex_trends_typology
        .rename(columns = {'type': 'category'})
        .assign(growth = lambda df: df.growth / 100)
    ),
    x_limit=300,
    y_limit= 1.2,
    mid_point=mid_point,
    baseline_growth=0,
    text_column = "category",
    values_label = "Average number of new publications per year",
)
fig.display()
fig.save(FIGURE_DIR / 'Figure7_openalex_trends.png', scale_factor=2.0)

alt.LayerChart(...)

In [129]:
patent_trends_typology = chart_trends.estimate_trend_type(
    patent_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)[['type', 'magnitude', 'growth', 'trend_type_suggestion']]
patent_trends_typology

,type,magnitude,growth,trend_type_suggestion
0,Biosciences,17.0,177.272727,emerging
1,Child care & preschool,34.6,-23.622047,dormant
2,Development & learning,52.0,0.714286,hot
4,Health,184.0,5.482042,hot
5,Parenting,52.2,-28.491620,stable
6,Society,0.4,inf,emerging


In [130]:
chart_trends._epsilon = 0.075
mid_point = patent_trends_typology.magnitude.median()

fig = chart_trends.mangitude_vs_growth_chart(
    data = (
        patent_trends_typology
        .rename(columns = {'type': 'category'})
        .assign(growth = lambda df: df.growth / 100)
    ),
    x_limit=300,
    y_limit= 2,
    mid_point=mid_point,
    baseline_growth=0,
    text_column = "category",
    values_label = "Average number of new patent applications per year",
)
fig.display()
fig.save(FIGURE_DIR / 'Figure7_patent_trends.png', scale_factor=2.0)

alt.LayerChart(...)

In [131]:
crunchbase_trends_typology = chart_trends.estimate_trend_type(
    crunchbase_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)[['type', 'magnitude', 'growth', 'trend_type_suggestion']]
crunchbase_trends_typology

,type,magnitude,growth,trend_type_suggestion
0,Biosciences,1805.317367,1748.296224,emerging
1,Child care & preschool,92699.788677,-0.863038,stable
2,Development & learning,291910.655018,44.765309,hot
4,Health,124167.480908,40.466729,hot
5,Parenting,54697.126335,-7.308249,dormant
6,Society,33146.601389,22.534738,emerging


In [132]:
chart_trends._epsilon = 0.025
mid_point = crunchbase_trends_typology.magnitude.median()

fig = chart_trends.mangitude_vs_growth_chart(
    data = (
        crunchbase_trends_typology
        .rename(columns = {'type': 'category'})
        .assign(growth = lambda df: df.growth / 100)
    ),
    x_limit=600000,
    y_limit= 1,
    mid_point=mid_point,
    baseline_growth=0,
    text_column = "category",
    values_label = "Average new venture funding per year (£ thousands)",
)
fig.display()
fig.save(FIGURE_DIR / 'Figure7_patent_trends.png', scale_factor=2.0)

alt.LayerChart(...)